In [14]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

from nd_aligner.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from nd_aligner.config.utils.io import load_config
from nd_aligner.models.ndaligner import init_nd_aligner_training_module

## INIT Models

In [15]:
# VCTK Only model

aligner_training_module_cfg_path = "/home/blue2959/monotonic_tts/checkpoints/ndaligner/v2.3/VCTK/model_config.json"
aligner_training_module_ckpt_path = "/home/blue2959/monotonic_tts/checkpoints/ndaligner/v2.3/VCTK/best_step_timit_bae_0.017094_step_143000_epoch_39.pth"

In [ ]:
# VCTK + LibriSpeech

aligner_training_module_cfg_path = "/home/blue2959/monotonic_tts/checkpoints/ndaligner/v2.3/VCTK+LibriSpeech/model_config.json"
aligner_training_module_ckpt_path = "/home/blue2959/monotonic_tts/checkpoints/ndaligner/v2.3/VCTK+LibriSpeech/best_step_timit_bae_0.017166_step_267000_epoch_9.pth"

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# aligner_training_module_cfg_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/TCD_rf_sweep/nd_aligner_vctk_20260821-215400/model_config.json"
# aligner_training_module_ckpt_path = "/shared/data_zfs/blue2959/ND_Aligner/experiments/v2.2/runs/TCD_rf_sweep/nd_aligner_vctk_20260821-215400/checkpoints_timit_bae/best_step_timit_bae_0.020831_step_139000_epoch_38.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in /home/blue2959/monotonic_tts/checkpoints/ndaligner/v2.3/VCTK/best_step_timit_bae_0.017094_step_143000_epoch_39.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [17]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [18]:
from nd_aligner.benchmark.timit.benchmarker import TIMITBenchMarker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
assert aligner.input_maker is not None

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
    boundary_mode="both",
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [19]:
with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        max_test_samples=None,
    )


Computing Alignments: 100%|██████████| 1680/1680 [00:57<00:00, 29.00it/s]


In [20]:
print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")
print(f"{metrics.coverage_ratio * 100.0:.2f} %")

17.34 ms
98.64 %
93.45 %
79.24 %
50.59 %
98.02 %


## INIT BenchMarkers (Buckeye)

In [9]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [21]:
from nd_aligner.benchmark.timit.benchmarker import TIMITBenchMarker
assert aligner.input_maker is not None

# go here and run this: ./benchmark/buckeye/buckeye_to_timit_eval_format.py
BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [22]:
with torch.inference_mode():
    metrics = buckeye_benchmarker.__call__(
        aligner=aligner,
        max_test_samples=None,
    )

Computing Alignments:   0%|          | 0/19273 [00:00<?, ?it/s]

Computing Alignments: 100%|██████████| 19273/19273 [12:47<00:00, 25.10it/s]


In [23]:
print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")
print(f"{metrics.coverage_ratio * 100.0:.2f} %")

28.85 ms
93.97 %
87.21 %
71.77 %
42.32 %
98.17 %
